# 03 — Train & Evaluate on Local GPU

Melatih baseline (TF-IDF+LR, TextCNN, BiLSTM, IndoBERTweet hard MTL, IndoBERTweet soft MTL) di GPU lokal.
Hasil: checkpoint, prediksi CSV, metrik CSV, confusion/PR/reliability PNG di `artifacts/`.

In [ ]:
from pathlib import Path
import sys
import json
import time

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, average_precision_score
from torch.utils.data import DataLoader

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from discourse.utils import load_yaml, set_seed, get_device, ensure_dir
from discourse.train import run_training
from discourse.predict import load_checkpoint, predict_dataframe
from discourse.evaluate import export_metrics_and_figures, select_thresholds
from discourse.calibration import calibrate_report
from discourse.datasets import DiscourseDataset, collate_batch

device = get_device("cuda")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
all_df = pd.read_parquet(ROOT / "data/processed/all_splits.parquet")
train_df = all_df[all_df.split == "train"].copy()
val_df = all_df[all_df.split == "validation"].copy()
test_df = all_df[all_df.split == "test"].copy()
print({k: len(v) for k, v in {"train": train_df, "val": val_df, "test": test_df}.items()})

## E0 — TF-IDF + Logistic Regression (CPU sanity baseline)

In [ ]:
def run_tfidf_lr(train_df, val_df, test_df, task_col_hard, task_name):
    tr = train_df[train_df[task_col_hard].notna()].copy()
    va = val_df[val_df[task_col_hard].notna()].copy()
    te = test_df[test_df[task_col_hard].notna()].copy()
    vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)
    Xtr = vec.fit_transform(tr["text_proc"].astype(str))
    Xva = vec.transform(va["text_proc"].astype(str))
    Xte = vec.transform(te["text_proc"].astype(str))
    clf = LogisticRegression(max_iter=200, class_weight="balanced", n_jobs=-1)
    clf.fit(Xtr, tr[task_col_hard].astype(int))
    # threshold on val macro-F1
    pva = clf.predict_proba(Xva)[:, 1]
    best_t, best_f1 = 0.5, -1
    yva = va[task_col_hard].astype(int).to_numpy()
    for t in np.linspace(0.1, 0.9, 17):
        f1 = f1_score(yva, (pva >= t).astype(int), average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    pte = clf.predict_proba(Xte)[:, 1]
    yte = te[task_col_hard].astype(int).to_numpy()
    pred = (pte >= best_t).astype(int)
    return {
        "experiment_id": "E0_tfidf_lr",
        "task": task_name,
        "split": "test",
        "n": int(len(te)),
        "threshold": best_t,
        "macro_f1": float(f1_score(yte, pred, average="macro", zero_division=0)),
        "pos_f1": float(f1_score(yte, pred, pos_label=1, zero_division=0)),
        "average_precision": float(average_precision_score(yte, pte)) if len(np.unique(yte)) > 1 else None,
    }

e0_rows = [
    run_tfidf_lr(train_df, val_df, test_df, "hard_tox", "tox"),
    run_tfidf_lr(train_df, val_df, test_df, "hard_pol", "pol"),
]
e0_df = pd.DataFrame(e0_rows)
ensure_dir(ROOT / "artifacts/metrics")
e0_df.to_csv(ROOT / "artifacts/metrics/E0_tfidf_lr_test_metrics.csv", index=False)
display(e0_df)

## Deep models on GPU (E1a TextCNN, E1b BiLSTM, E1c/E2 IndoBERTweet hard MTL, E3 soft MTL)

In [ ]:
exp_configs = [
    "configs/textcnn.yaml",
    "configs/bilstm.yaml",
    "configs/indobertweet.yaml",
    "configs/multitask_soft.yaml",
]

all_metrics = [e0_df]
run_summaries = []

for cfg_path in exp_configs:
    cfg = load_yaml(ROOT / cfg_path)
    exp_id = cfg["experiment_id"]
    out_dir = ROOT / "artifacts/checkpoints" / exp_id
    print("\n=== Training", exp_id, "===")
    t0 = time.time()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    result = run_training(cfg, train_df, val_df, out_dir=out_dir, device_prefer="cuda")
    elapsed = time.time() - t0
    peak_vram = None
    if torch.cuda.is_available():
        peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print(f"done {exp_id} in {elapsed/60:.1f} min | best={result['best_score']:.4f} | peak_vram_mb={peak_vram}")

    model, cfg_loaded, thresholds, vocab, tokenizer, device = load_checkpoint(result["best_path"], "cuda")
    # refresh thresholds on validation
    val_pred = predict_dataframe(model, val_df, cfg_loaded, vocab, tokenizer, device)
    thresholds = select_thresholds(val_pred)
    test_pred = predict_dataframe(model, test_df, cfg_loaded, vocab, tokenizer, device)
    test_pred["experiment_id"] = exp_id
    test_pred["split"] = "test"

    metrics_df = export_metrics_and_figures(
        exp_id,
        test_pred,
        thresholds,
        split_name="test",
        metrics_dir=ROOT / "artifacts/metrics",
        figures_dir=ROOT / "artifacts/figures",
    )
    cal_df = calibrate_report(
        test_pred,
        exp_id,
        figures_dir=ROOT / "artifacts/figures",
        tables_dir=ROOT / "artifacts/tables",
    )
    all_metrics.append(metrics_df)
    run_summaries.append({
        "experiment_id": exp_id,
        "device": result["device"],
        "best_val_mean_macro_f1": result["best_score"],
        "train_minutes": elapsed / 60,
        "peak_vram_mb": peak_vram,
        "thresholds": json.dumps(thresholds),
    })
    display(metrics_df)
    display(cal_df)

summary_df = pd.DataFrame(run_summaries)
summary_df.to_csv(ROOT / "artifacts/metrics/training_summary.csv", index=False)
combined = pd.concat(all_metrics, ignore_index=True)
combined.to_csv(ROOT / "artifacts/metrics/all_experiments_test_metrics.csv", index=False)
display(summary_df)
display(combined)

In [ ]:
from IPython.display import Image, display

print("Key local artifacts:")
for pattern in [
    "artifacts/metrics/*.csv",
    "artifacts/predictions/*.csv",
    "artifacts/figures/*_cm.png",
    "artifacts/figures/*_pr.png",
    "artifacts/figures/*_reliability.png",
]:
    files = sorted(ROOT.glob(pattern))
    print(f"\n{pattern}: {len(files)} files")
    for f in files[:12]:
        print(" -", f.relative_to(ROOT))

# show a few figures
for f in sorted((ROOT / "artifacts/figures").glob("E1c*_test_tox_cm.png"))[:1]:
    display(Image(filename=str(f)))
for f in sorted((ROOT / "artifacts/figures").glob("E1c*_tox_reliability.png"))[:1]:
    display(Image(filename=str(f)))